In [ ]:
## data acquisition- download data from LIGO

## downloads gravitational wave events and background noise events from LIGO open science center

In [ ]:
import os
import numpy as np
from gwpy.timeseries import TimeSeries
from gwosc import datasets
from datetime import datetime

BASE_DIR = r'C:\Users\srika\gw_wd'
EVENTS_DIR = os.path.join(BASE_DIR, 'data', 'events')
NOISE_DIR = os.path.join(BASE_DIR, 'data', 'noise')


print("PATH VERIFICATION")

print(f"\nBase directory  : {BASE_DIR}")
print(f"Events directory: {EVENTS_DIR}")
print(f"Noise directory : {NOISE_DIR}")

os.makedirs(EVENTS_DIR, exist_ok=True)
os.makedirs(NOISE_DIR, exist_ok=True)

print(f"\n Base exists  : {os.path.exists(BASE_DIR)}")
print(f" Events exists: {os.path.exists(EVENTS_DIR)}")
print(f" Noise exists : {os.path.exists(NOISE_DIR)}")

print("\n All paths verified! Ready to download.")

In [ ]:
def download_event(event_name, detectors=['H1', 'L1']):
    """Download a gravitational wave event using absolute paths"""
    
    try:
        gps = datasets.event_gps(event_name)
        
        print(f"\n Event   : {event_name}")
        print(f"\n GPS Time: {gps}")
        
        
        for detector in detectors:
            try:
                print(f"\n  Downloading {detector}...", end=' ', flush=True)
                
                data = TimeSeries.fetch_open_data(
                    detector,
                    gps - 16,
                    gps + 16,
                    cache=True
                )
                
                # Absolute paths
                filepath = os.path.join(EVENTS_DIR, f'{event_name}_{detector}.npy')
                metapath = os.path.join(EVENTS_DIR, f'{event_name}_{detector}_meta.npy')
                
                np.save(filepath, data.value)
                
                metadata = {
                    'sample_rate': float(data.sample_rate.value),
                    'gps_time': gps,
                    'detector': detector,
                    'duration': 32,
                    'event_name': event_name,
                    'download_date': datetime.now().isoformat()
                }
                np.save(metapath, metadata)
                
                # Verify saved
                if os.path.exists(filepath):
                    size = os.path.getsize(filepath) / 1024 / 1024
                    print(f" Saved! ({len(data):,} samples, {size:.2f} MB)")
                    print(f"     {filepath}")
                else:
                    print(f" File not saved!")
                    
            except Exception as e:
                print(f" Error: {str(e)[:60]}")
        
        return True
        
    except Exception as e:
        print(f"\n Failed: {e}")
        return False


def download_noise_segment(index, gps_time, detectors=['H1', 'L1']):
    """Download background noise using absolute paths"""
    
    for detector in detectors:
        try:
            data = TimeSeries.fetch_open_data(
                detector,
                gps_time,
                gps_time + 32,
                cache=True
            )
            
            filepath = os.path.join(NOISE_DIR, f'noise_{index:03d}_{detector}.npy')
            metapath = os.path.join(NOISE_DIR, f'noise_{index:03d}_{detector}_meta.npy')
            
            np.save(filepath, data.value)
            
            metadata = {
                'sample_rate': float(data.sample_rate.value),
                'gps_time': gps_time,
                'detector': detector,
                'duration': 32,
                'segment_index': index
            }
            np.save(metapath, metadata)
            
        except Exception as e:
            pass

print(" Download functions ready!")

events_to_download = [
    'GW150914',  # First detection ever
    'GW151226',  # Second detection
    'GW170817',  # Neutron star merger
    'GW170814',  # Three-detector observation
]


print(" DOWNLOADING GRAVITATIONAL WAVE EVENTS")

print(f"Events to download: {len(events_to_download)}")
print("  Estimated time : 3-5 minutes")
print("  Please wait, do not close Jupyter...\n")

success_count = 0
for event in events_to_download:
    if download_event(event):
        success_count += 1


print(f"\n Downloaded: {success_count}/{len(events_to_download)} events")


In [ ]:
#noise events

noise_gps_times = [
    1126200000,
    1126400000,
    1127000000,
    1128000000,
    1129000000,
    1130000000,
    1131000000,
    1132000000,
    1133000000,
    1134000000,
]


print(" DOWNLOADING NOISE SEGMENTS")

print(f"Segments to download: {len(noise_gps_times)}")
print("  Estimated time  : 2-3 minutes")
print("  Please wait, do not close Jupyter...\n")

for i, gps in enumerate(noise_gps_times):
    print(f"  Segment {i+1:2d}/{len(noise_gps_times)}... ", end='', flush=True)
    download_noise_segment(i, gps)
    


print(f"\n All noise segments downloaded!")



In [ ]:
#verification

import glob


print(" FINAL VERIFICATION\n")


# Check events
event_files = [f for f in glob.glob(os.path.join(EVENTS_DIR, '*.npy')) 
               if 'meta' not in f]

print(f"\n Events folder:")
print(f"   Files found: {len(event_files)}")
for f in sorted(event_files):
    size = os.path.getsize(f) / 1024 / 1024
    print(f"    {os.path.basename(f):35s} {size:.2f} MB")

# Check noise
noise_files = [f for f in glob.glob(os.path.join(NOISE_DIR, '*.npy'))
               if 'meta' not in f]
print(f" Noise folder:")
print(f"   Files found: {len(noise_files)}")
for f in sorted(noise_files):
    size = os.path.getsize(f) / 1024 / 1024
    print(f"    {os.path.basename(f):35s} {size:.2f} MB")

# Total size
all_files = event_files + noise_files
total_size = sum(os.path.getsize(f) for f in all_files) / 1024 / 1024
print(f"\n Total size: {total_size:.1f} MB")

# Quick load test
print("\n Quick Load Test:")
test_data = np.load(event_files[0])
test_meta = np.load(event_files[0].replace('.npy', '_meta.npy'), 
                    allow_pickle=True).item()

print(f"   File       : {os.path.basename(event_files[0])}")
print(f"   Samples    : {len(test_data):,}")
print(f"   Sample rate: {test_meta['sample_rate']} Hz")
print(f"   Duration   : {test_meta['duration']} seconds")
print(f"   Data range : [{test_data.min():.2e}, {test_data.max():.2e}]")


print(f" DATA ACQUISITION 100% COMPLETE!\n")

print(f"\nYou now have:")
print(f"   {len(event_files)} gravitational wave event files")
print(f"   {len(noise_files)} background noise files")
print(f"   All data verified and ready!")
print(f"\n Next step: Phase 2 - Data Visualization!")